
# Recertification KYC — Pipeline d'extraction (Pièce d'identité + Justificatif de domicile)

**Objectif** : à partir d'un ZIP contenant un dossier par client (nommé par son ID), extraire
et structurer les informations de deux documents (`JUSTIFICATIF IDENTITE.PDF` et
`JUSTIFICATIF DOMICILE.PDF`), quels que soient le pays d'émission, la qualité du scan
(photocopie, scan de travers, rotation aléatoire) et la langue (mix arabe/français pour
les documents algériens).

## Architecture retenue

1. **Dézippage + repérage robuste des 2 PDF cibles** par client (matching flou sur le nom de fichier).
2. **PDF → image** (haute résolution) puis **prétraitement** : correction de rotation (0/90/180/270°
   via Tesseract OSD) + **deskew** fin (angle résiduel via OpenCV).
3. **Détection MRZ** (Machine Readable Zone) sur les cartes d'identité :
   - Si MRZ présente → OCR de la zone + parsing structuré avec la librairie `mrz`
     (TD1/TD2/TD3) → source **fiable et déterministe** pour nom, prénom, n° pièce,
     nationalité, date de naissance, sexe, date d'expiration.
   - Si pas de MRZ → on s'appuie entièrement sur le modèle VLM.
4. **Détection du pays d'émission** (Algérie vs autre) via le code pays MRZ (`DZA`) ou
   des heuristiques texte (présence d'arabe, mentions "République Algérienne", wilaya, etc.).
5. **Sélection du prompt / schéma** :
   - Documents **algériens** → schéma JSON strict par type de document (CNI/CIN algérienne,
     certificat/attestation de résidence), prompt bilingue AR/FR.
   - Documents **non-algériens** → prompt générique (schéma libre) car formats trop hétérogènes.
6. **Extraction VLM** : le modèle (au choix parmi les 3 fournis) reçoit l'image prétraitée
   + le prompt, et renvoie un JSON structuré. La sortie MRZ (si dispo) est fusionnée/arbitrée
   avec la sortie VLM (la MRZ prime sur les champs qu'elle couvre).
7. **Consolidation** en DataFrame (1 ligne / client / document).
8. **Réconciliation** avec `tiers.csv` : normalisation des chaînes (accents, casse, arabe→latin
   quand transcrit) puis comparaison floue (`rapidfuzz`) champ par champ pour détecter les
   incohérences (nom, prénom, date de naissance, n° de pièce, adresse...).

## Pourquoi cette approche plutôt qu'un OCR classique seul ?

- Un OCR classique (Tesseract) est très fragile sur des photocopies de mauvaise qualité,
  du texte manuscrit, ou du mix arabe/français dans le même document.
- Un **VLM (Vision-Language Model)** comprend le *layout* du document (où se trouve le nom,
  l'adresse, etc.) même sur un scan dégradé ou pivoté, et peut raisonner en zero/few-shot
  sur des formats de pays différents.
- La **MRZ**, quand elle existe, est une source normalisée internationalement (ICAO 9303) :
  on la privilégie car elle est beaucoup plus fiable qu'un VLM sur les champs qu'elle couvre
  (nom, prénom, n° document, dates, nationalité), et elle sert aussi de **contrôle qualité**
  (checksum intégré) pour valider/invalider l'extraction VLM.
- Cette combinaison **MRZ (déterministe) + VLM (généraliste) + schéma conditionnel par pays**
  est le meilleur compromis précision/couverture pour un portefeuille multi-pays.


## 0. Installation des dépendances (versions figées)

In [ ]:

# À exécuter une seule fois dans l'environnement du notebook.
# Adapter selon que le poste dispose déjà d'un torch+CUDA installé (ne pas réinstaller torch dans ce cas).

%pip install --quiet \
    pymupdf==1.24.10 \
    Pillow==10.4.0 \
    opencv-python-headless==4.10.0.84 \
    numpy==1.26.4 \
    pandas==2.2.2 \
    pytesseract==0.3.13 \
    mrz==0.6.2 \
    rapidfuzz==3.9.6 \
    Unidecode==1.3.8 \
    transformers==4.49.0 \
    accelerate==0.34.2 \
    qwen-vl-utils==0.0.8 \
    torch==2.4.0 \
    einops==0.8.0 \
    timm==1.0.9

# Binaire système requis par pytesseract (à installer via apt si absent) :
#   sudo apt-get install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-fra


## 1. Imports

In [ ]:

import os
import re
import io
import json
import zipfile
import shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import cv2
from PIL import Image

import pytesseract
from rapidfuzz import fuzz
from unidecode import unidecode

from mrz.checker.td1 import TD1CodeChecker
from mrz.checker.td2 import TD2CodeChecker
from mrz.checker.td3 import TD3CodeChecker


## 2. Configuration

In [ ]:

CONFIG = {
    # --- Entrées ---
    "zip_path": "/path/to/dossiers_clients.zip",
    "tiers_csv_path": "/path/to/tiers.csv",
    "work_dir": "/home/claude/kyc_work",       # dossier de travail temporaire (extraction du zip)
    "output_dir": "/home/claude/kyc_output",   # résultats

    # --- Noms cibles des PDF à repérer dans chaque dossier client (matching flou) ---
    "target_docs": {
        "identite": "JUSTIFICATIF IDENTITE",
        "domicile": "JUSTIFICATIF DOMICILE",
    },
    "filename_match_threshold": 80,  # score rapidfuzz (0-100) pour accepter un match de nom de fichier

    # --- Modèle VLM à utiliser : "qwen25vl" | "internvl3" | "paddleocr_vl" ---
    "model_choice": "qwen25vl",
    "model_paths": {
        "qwen25vl": "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main",
        "paddleocr_vl": "/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main",
        "internvl3": "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/internVL3-8B/main",
    },

    # --- Rendu PDF -> image ---
    "pdf_dpi": 300,

    # --- Réconciliation ---
    "fuzzy_match_threshold": 88,  # en dessous -> incohérence signalée

    # Mapping des colonnes de tiers.csv vers les champs extraits (à adapter à votre fichier réel)
    "tiers_columns_mapping": {
        "id_tiers": "id_tiers",
        "nom": "nom",
        "prenom": "prenom",
        "date_naissance": "date_naissance",
        "numero_piece": "numero_piece_identite",
        "adresse": "adresse",
        "pays": "pays",
    },
}

os.makedirs(CONFIG["work_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)


## 3. Dézippage et repérage des 2 PDF cibles par client

Chaque dossier client est nommé par l'ID du tiers. On tolère des variations dans le nommage
des fichiers (accents, espaces/underscores, casse, extensions) grâce à un score de similarité
`rapidfuzz` plutôt qu'une égalité stricte.

In [ ]:

def extract_zip(zip_path: str, dest_dir: str) -> str:
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    return dest_dir


def _normalize_filename(name: str) -> str:
    name = Path(name).stem
    name = unidecode(name).upper()
    name = re.sub(r"[_\-\.]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def find_target_pdfs(client_dir: Path, target_docs: Dict[str, str], threshold: int) -> Dict[str, Optional[Path]]:
    '''Retourne {'identite': Path|None, 'domicile': Path|None} en matchant les noms de fichiers du
    dossier client contre les libelles cibles, de maniere tolerante aux variations d'orthographe.'''
    found = {key: None for key in target_docs}
    pdf_files = [p for p in client_dir.rglob("*.pdf")] + [p for p in client_dir.rglob("*.PDF")]
    pdf_files = list(set(pdf_files))

    for key, target_label in target_docs.items():
        best_score, best_path = 0, None
        target_norm = _normalize_filename(target_label)
        for pdf in pdf_files:
            score = fuzz.token_sort_ratio(target_norm, _normalize_filename(pdf.name))
            if score > best_score:
                best_score, best_path = score, pdf
        if best_score >= threshold:
            found[key] = best_path
    return found


def list_client_folders(root_dir: str) -> List[Path]:
    root = Path(root_dir)
    return [p for p in root.iterdir() if p.is_dir()]


## 4. PDF → images + prétraitement (rotation & deskew)

- Conversion haute résolution avec PyMuPDF (pas de dépendance à `poppler`).
- Correction de rotation grossière (0/90/180/270°) via Tesseract OSD (`image_to_osd`),
  robuste même sur du texte dégradé.
- Deskew fin (quelques degrés résiduels, typiques d'un scan/photo un peu de travers)
  via `minAreaRect` sur l'image binarisée.

In [ ]:

def pdf_to_images(pdf_path: Path, dpi: int = 300) -> List[Image.Image]:
    images = []
    doc = fitz.open(pdf_path)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        images.append(img)
    doc.close()
    return images


def correct_coarse_rotation(pil_img: Image.Image) -> Image.Image:
    '''Corrige les rotations de 90/180/270 degres via Tesseract OSD.'''
    try:
        osd = pytesseract.image_to_osd(pil_img, output_type=pytesseract.Output.DICT)
        angle = int(osd.get("rotate", 0))
    except pytesseract.TesseractError:
        angle = 0
    if angle != 0:
        pil_img = pil_img.rotate(-angle, expand=True, fillcolor=(255, 255, 255))
    return pil_img


def deskew_fine(pil_img: Image.Image) -> Image.Image:
    '''Corrige un angle residuel (quelques degres) via minAreaRect OpenCV.'''
    img = np.array(pil_img.convert("L"))
    _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(thresh > 0))
    if coords.shape[0] < 50:
        return pil_img  # pas assez de contenu pour estimer un angle fiable
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.3 or abs(angle) > 15:
        return pil_img  # ignore les angles négligeables ou aberrants (déjà géré par OSD sinon)
    (h, w) = img.shape
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(np.array(pil_img), M, (w, h), flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
    return Image.fromarray(rotated)


def preprocess_document_image(pil_img: Image.Image) -> Image.Image:
    img = correct_coarse_rotation(pil_img)
    img = deskew_fine(img)
    return img


## 5. Détection du type de document (en-tête) + détection "biométrique" + parsing MRZ

Trois étapes, comme demandé :

1. **Type de document** : on OCRise uniquement l'en-tête (tiers supérieur de l'image) et on
   cherche des mots-clés ("Nom", "Prénom(s)", "République Algérienne...", "Carte d'identité", ...)
   pour confirmer qu'il s'agit bien d'une pièce d'identité.
2. **Carte biométrique ou non** : une carte est considérée comme *biométrique* si une **zone MRZ**
   (3 lignes de type ICAO en bas du document) est détectée. Si aucune ligne MRZ n'est trouvée
   → carte non biométrique (ancien format) → extraction 100% VLM.
3. **Parsing MRZ** :
   - Pour les **cartes d'identité algériennes** (ligne 1 de la MRZ commençant par `ID` + code
     pays `DZA`), on applique un **parseur dédié**, volontairement basé sur des règles simples
     et non sur un contrôle de checksum strict (les scans/photocopies dégradent souvent les
     chiffres de contrôle MRZ alors que les champs eux-mêmes restent lisibles) :
       - ligne 2, positions 0-5 → **date de naissance** au format `YYMMDD` ;
       - dans la ligne 2, on repère le premier caractère de sexe (`M`, `F` ou `<`) qui suit la
         date de naissance, puis les **6 caractères suivants** → **date d'expiration** `YYMMDD` ;
       - ligne 3 : texte avant `<<` → **nom**, texte après `<<` → **prénom** ;
       - les deux dates `YYMMDD` sont ensuite normalisées en `yyyy-mm-dd`.
   - Pour les **autres documents** (passeports, cartes d'identité d'autres pays), on retombe sur
     un parsing générique **avec contrôle de checksum ICAO** (TD1/TD2/TD3, librairie `mrz`), plus
     strict car on n'a pas de règles métier spécifiques à ces formats.

In [ ]:

# --- 5.1 Détection du type de document via l'en-tête -----------------------------------------

ID_CARD_HEADER_KEYWORDS = [
    "NOM", "PRENOM", "CARTE NATIONALE D'IDENTITE", "CARTE D'IDENTITE",
    "NATIONAL IDENTITY CARD", "IDENTITY CARD", "REPUBLIQUE ALGERIENNE",
    "PIECE D'IDENTITE", "بطاقة التعريف", "الجمهورية الجزائرية",
]


def detect_document_kind_from_header(pil_img: Image.Image) -> bool:
    '''Retourne True si l'en-tete du document (tiers superieur) evoque une piece d'identite.'''
    w, h = pil_img.size
    header_crop = pil_img.crop((0, 0, w, int(h * 0.35)))
    try:
        text = pytesseract.image_to_string(header_crop, lang="fra+ara")
    except pytesseract.TesseractError:
        text = pytesseract.image_to_string(header_crop, lang="fra")
    text_norm = unidecode(text).upper()
    return any(unidecode(kw).upper() in text_norm for kw in ID_CARD_HEADER_KEYWORDS)


# --- 5.2 Extraction des lignes MRZ candidates -------------------------------------------------

MRZ_LINE_RE = re.compile(r"^[A-Z0-9<]{28,45}$")


def _extract_mrz_candidate_lines(pil_img: Image.Image) -> List[str]:
    w, h = pil_img.size
    bottom_crop = pil_img.crop((0, int(h * 0.6), w, h))  # zone basse = MRZ probable
    ocr_text = pytesseract.image_to_string(
        bottom_crop, lang="eng", config="--psm 6 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789<"
    )
    lines = [l.strip().replace(" ", "") for l in ocr_text.splitlines() if l.strip()]
    return [l for l in lines if MRZ_LINE_RE.match(l)]


# --- 5.3 Conversion YYMMDD -> yyyy-mm-dd --------------------------------------------------------

def yymmdd_to_iso(yymmdd: str, is_birth_date: bool) -> Optional[str]:
    if not yymmdd or len(yymmdd) != 6 or not yymmdd.isdigit():
        return None
    yy, mm, dd = yymmdd[0:2], yymmdd[2:4], yymmdd[4:6]
    yy_int = int(yy)
    current_yy = int(str(pd.Timestamp.now().year)[-2:])
    if is_birth_date:
        # heuristique standard MRZ : une naissance ne peut pas etre "dans le futur"
        century = 1900 if yy_int > current_yy + 1 else 2000
    else:
        # une date d'expiration de document est toujours recente/future -> 20yy
        century = 2000
    year = century + yy_int
    if not (1 <= int(mm) <= 12 and 1 <= int(dd) <= 31):
        return None
    return f"{year:04d}-{mm}-{dd}"


# --- 5.4 Parseur dedie aux CNI biometriques algeriennes (regles metier, sans checksum) --------

def find_algerian_mrz_block(candidate_lines: List[str]) -> Optional[List[str]]:
    '''Repere le bloc de 3 lignes MRZ dont la 1ere ligne commence par ID + code pays DZA.'''
    for i, line in enumerate(candidate_lines):
        if re.match(r"^ID[A-Z]{3}", line) and "DZA" in line[:6] and i + 2 < len(candidate_lines):
            return candidate_lines[i:i + 3]
    return None


def parse_algerian_id_mrz(lines: List[str]) -> Optional[Dict[str, Any]]:
    '''Applique les regles metier fournies pour les CNI biometriques algeriennes (format TD1).'''
    if len(lines) < 3:
        return None
    line1, line2, line3 = lines[0], lines[1], lines[2]

    # Date de naissance : 6 premiers caracteres de la ligne 2
    birth_raw = line2[0:6]
    birth_iso = yymmdd_to_iso(birth_raw, is_birth_date=True)

    # Date d'expiration : 6 caracteres suivant le premier marqueur de sexe (M/F/<) apres la date de naissance
    sex, expiry_iso = None, None
    sex_match = re.search(r"[MF<]", line2[6:16])
    if sex_match:
        sex_pos = 6 + sex_match.start()
        sex = sex_match.group(0)
        expiry_raw = line2[sex_pos + 1: sex_pos + 7]
        expiry_iso = yymmdd_to_iso(expiry_raw, is_birth_date=False)

    # Nom / prenom : ligne 3, separes par '<<'
    surname, given_names = None, None
    if "<<" in line3:
        parts = line3.split("<<")
        surname = parts[0].replace("<", " ").strip()
        given_names = (parts[1] if len(parts) > 1 else "").replace("<", " ").strip()

    # Numero de document : ligne 1, apres 'ID' + code pays
    document_number = None
    doc_match = re.match(r"^ID[A-Z]{3}([A-Z0-9]+)<*", line1)
    if doc_match:
        document_number = doc_match.group(1)

    return {
        "surname": surname or None,
        "given_names": given_names or None,
        "birth_date": birth_iso,
        "sex": sex if sex in ("M", "F") else None,
        "expiry_date": expiry_iso,
        "document_number": document_number,
        "nationality": "DZA",
        "country": "DZA",
        "optional_data": None,
    }


# --- 5.5 Normalisation des dates pour le parsing generique international (checksum ICAO) ------

def _format_generic_mrz_date(value, is_birth: bool) -> Optional[str]:
    if value is None:
        return None
    if hasattr(value, "isoformat"):
        return value.isoformat()
    value = str(value)
    if len(value) == 6 and value.isdigit():
        return yymmdd_to_iso(value, is_birth_date=is_birth)
    return value


# --- 5.6 Orchestrateur : biometrique ? algerien (regles dediees) ou generique (checksum) ? ----

def detect_and_parse_mrz(pil_img: Image.Image) -> Dict[str, Any]:
    '''Retourne mrz_present, is_biometric, mrz_type, fields, checksum_valid, raw_lines.'''
    candidate_lines = _extract_mrz_candidate_lines(pil_img)
    result = {
        "mrz_present": False,
        "is_biometric": False,
        "mrz_type": None,
        "fields": None,
        "checksum_valid": None,
        "raw_lines": candidate_lines,
    }

    if len(candidate_lines) < 2:
        return result  # pas de MRZ detectee -> document non biometrique -> extraction 100% VLM

    result["is_biometric"] = True

    # --- Cas carte d'identite algerienne : regles metier dediees ---
    algerian_block = find_algerian_mrz_block(candidate_lines)
    if algerian_block:
        parsed = parse_algerian_id_mrz(algerian_block)
        if parsed:
            result.update({"mrz_present": True, "mrz_type": "TD1_DZ", "fields": parsed})
            # controle de coherence optionnel (informatif uniquement, n'invalide pas le parsing)
            try:
                checker = TD1CodeChecker("\n".join(algerian_block))
                result["checksum_valid"] = bool(checker.result)
            except Exception:
                result["checksum_valid"] = False
            return result

    # --- Cas generique (passeports, cartes d'autres pays) : checksum ICAO obligatoire ---
    attempts = [
        (3, TD1CodeChecker, "TD1"),  # carte d'identite : 3 lignes de 30 caracteres
        (2, TD2CodeChecker, "TD2"),  # 2 lignes de 36
        (2, TD3CodeChecker, "TD3"),  # passeport : 2 lignes de 44
    ]
    for n_lines, checker_cls, label in attempts:
        if len(candidate_lines) < n_lines:
            continue
        for start in range(0, len(candidate_lines) - n_lines + 1):
            block_lines = candidate_lines[start:start + n_lines]
            mrz_string = "\n".join(block_lines)
            try:
                checker = checker_cls(mrz_string)
                if checker.result:  # checksum global valide
                    f = checker.fields()
                    result.update({
                        "mrz_present": True,
                        "mrz_type": label,
                        "checksum_valid": True,
                        "fields": {
                            "surname": getattr(f, "surname", None),
                            "given_names": getattr(f, "name", None),
                            "document_number": getattr(f, "document_number", None),
                            "nationality": getattr(f, "nationality", None),
                            "country": getattr(f, "country", None),
                            "birth_date": _format_generic_mrz_date(getattr(f, "birth_date", None), is_birth=True),
                            "sex": getattr(f, "sex", None),
                            "expiry_date": _format_generic_mrz_date(getattr(f, "expiry_date", None), is_birth=False),
                            "optional_data": getattr(f, "optional_data", None),
                        },
                    })
                    return result
            except Exception:
                continue
    return result


## 6. Détection du pays d'émission (Algérie vs autre)

Priorité au code pays MRZ (`DZA`). À défaut, heuristique texte : présence de caractères arabes
et/ou de mentions caractéristiques ("République Algérienne", "Wilaya", "Commune de", ...).

In [ ]:

ARABIC_RE = re.compile(r"[\u0600-\u06FF]")
ALGERIA_KEYWORDS = [
    "REPUBLIQUE ALGERIENNE", "ALGERIE", "WILAYA", "COMMUNE DE", "DAIRA",
    "الجمهورية الجزائرية", "بطاقة التعريف", "ولاية",
]


def detect_country_is_algeria(pil_img: Image.Image, mrz_result: Dict[str, Any]) -> bool:
    if mrz_result.get("fields") and mrz_result["fields"].get("country"):
        if str(mrz_result["fields"]["country"]).upper().startswith("DZA"):
            return True

    # Fallback : OCR rapide multi-langue (fr+ar) sur l'image entière pour chercher des indices
    try:
        ocr_text = pytesseract.image_to_string(pil_img, lang="fra+ara")
    except pytesseract.TesseractError:
        ocr_text = pytesseract.image_to_string(pil_img, lang="fra")

    ocr_upper = unidecode(ocr_text).upper()
    if ARABIC_RE.search(ocr_text):
        return True
    for kw in ALGERIA_KEYWORDS:
        if unidecode(kw).upper() in ocr_upper:
            return True
    return False


## 7. Schémas d'extraction

- **Algérie** : schéma JSON strict par type de document (champs figés, prompt bilingue FR/AR).
- **Autres pays** : prompt "libre" (pas de schéma imposé, formats trop hétérogènes) — on demande
  au modèle de renvoyer les champs qu'il identifie, avec des clés génériques normalisées.

In [ ]:

SCHEMA_ID_ALGERIE = {
    "nom": "string",
    "prenom": "string",
    "date_naissance": "YYYY-MM-DD",
    "lieu_naissance": "string",
    "sexe": "M/F",
    "numero_piece": "string",
    "date_delivrance": "YYYY-MM-DD",
    "date_expiration": "YYYY-MM-DD",
    "autorite_delivrance": "string",
    "adresse": "string",
    "nom_pere": "string",
    "nom_mere": "string",
}

SCHEMA_DOMICILE_ALGERIE = {
    "nom": "string",
    "prenom": "string",
    "adresse": "string",
    "commune": "string",
    "wilaya": "string",
    "date_document": "YYYY-MM-DD",
    "type_document": "facture / attestation / autre",
    "emetteur": "string (ex: Sonelgaz, Algérie Télécom, APC...)",
}

GENERIC_SCHEMA = {
    "nom": "string",
    "prenom": "string",
    "date_naissance": "YYYY-MM-DD ou null",
    "numero_piece": "string ou null",
    "adresse": "string ou null",
    "pays": "string",
    "date_document": "YYYY-MM-DD ou null",
    "autres_champs_utiles": "objet libre {cle: valeur} pour tout champ pertinent non listé",
}


def build_prompt(doc_kind: str, is_algeria: bool) -> str:
    # doc_kind: 'identite' | 'domicile'
    if is_algeria and doc_kind == "identite":
        schema = SCHEMA_ID_ALGERIE
        contexte = ("Ce document est une pièce d'identité algérienne, potentiellement rédigée "
                    "en mixte arabe/français, parfois issue d'une photocopie de mauvaise qualité.")
    elif is_algeria and doc_kind == "domicile":
        schema = SCHEMA_DOMICILE_ALGERIE
        contexte = ("Ce document est un justificatif de domicile algérien (facture, attestation "
                    "communale, etc.), potentiellement en mixte arabe/français.")
    else:
        schema = GENERIC_SCHEMA
        contexte = ("Ce document (pièce d'identité ou justificatif de domicile) provient d'un pays "
                    "autre que l'Algérie. Le format n'est pas connu à l'avance : identifie et "
                    "extrais tous les champs pertinents.")

    prompt = f'''Tu es un expert en analyse de documents KYC (connaissance client).
{contexte}

Analyse attentivement l'image fournie (le document peut être scanné, photocopié, de qualité
moyenne, ou légèrement de travers) et extrais les informations suivantes au format JSON strict,
en respectant EXACTEMENT les clés ci-dessous (mets null si une information est illisible ou absente) :

{json.dumps(schema, ensure_ascii=False, indent=2)}

Consignes :
- Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant/après, sans balises markdown.
- Conserve les accents et l'orthographe exacte des noms.
- Les dates doivent être normalisées au format YYYY-MM-DD si possible.
- Si le document est bilingue (arabe/français), privilégie la transcription en caractères latins
  (français) pour les champs texte, sauf s'il n'existe pas d'équivalent latin lisible.
'''
    return prompt


## 8. Chargement du modèle VLM

Trois adaptateurs sont fournis pour les 3 modèles mis à disposition. Seul celui sélectionné
dans `CONFIG["model_choice"]` est chargé. Les API `transformers` diffèrent légèrement selon
le modèle : adapter au besoin selon la version exacte publiée sur le model hub interne.

In [ ]:

import torch

class VLMEngine:
    def __init__(self, model_choice: str, model_path: str):
        self.model_choice = model_choice
        self.model_path = model_path
        self.model = None
        self.processor = None
        self._load()

    def _load(self):
        if self.model_choice == "qwen25vl":
            from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                self.model_path, torch_dtype="auto", device_map="auto"
            )
            self.processor = AutoProcessor.from_pretrained(self.model_path)

        elif self.model_choice == "internvl3":
            from transformers import AutoModel, AutoTokenizer
            self.model = AutoModel.from_pretrained(
                self.model_path, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto"
            ).eval()
            self.processor = AutoTokenizer.from_pretrained(
                self.model_path, trust_remote_code=True, use_fast=False
            )

        elif self.model_choice == "paddleocr_vl":
            # API en évolution rapide côté PaddleOCR-VL : vérifier le README du model hub interne
            # (device/precision, méthode d'inférence dédiée type `.predict()` ou wrapper transformers).
            from transformers import AutoModel, AutoProcessor
            self.model = AutoModel.from_pretrained(
                self.model_path, trust_remote_code=True, device_map="auto"
            ).eval()
            self.processor = AutoProcessor.from_pretrained(self.model_path, trust_remote_code=True)

        else:
            raise ValueError(f"model_choice inconnu: {self.model_choice}")

    def generate(self, pil_img: Image.Image, prompt: str, max_new_tokens: int = 1024) -> str:
        if self.model_choice == "qwen25vl":
            from qwen_vl_utils import process_vision_info
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": pil_img},
                    {"type": "text", "text": prompt},
                ],
            }]
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                     padding=True, return_tensors="pt").to(self.model.device)
            generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
            trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
            output = self.processor.batch_decode(trimmed, skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=False)
            return output[0]

        elif self.model_choice == "internvl3":
            # Pattern standard InternVL (méthode .chat() fournie par trust_remote_code)
            pixel_values = self.model.load_image(pil_img) if hasattr(self.model, "load_image") else None
            generation_config = dict(max_new_tokens=max_new_tokens, do_sample=False)
            response = self.model.chat(self.processor, pixel_values, prompt, generation_config)
            return response

        elif self.model_choice == "paddleocr_vl":
            inputs = self.processor(images=pil_img, text=prompt, return_tensors="pt").to(self.model.device)
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
            return self.processor.batch_decode(out, skip_special_tokens=True)[0]


# vlm_engine = VLMEngine(CONFIG["model_choice"], CONFIG["model_paths"][CONFIG["model_choice"]])


## 9. Parsing robuste de la réponse JSON du modèle

In [ ]:

JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)

def safe_parse_json(raw_output: str) -> Dict[str, Any]:
    match = JSON_BLOCK_RE.search(raw_output)
    if not match:
        return {"_parse_error": True, "_raw_output": raw_output}
    candidate = match.group(0)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        # tentative de nettoyage simple (virgules traînantes, guillemets simples)
        cleaned = candidate.replace("'", '"')
        cleaned = re.sub(r",\s*}", "}", cleaned)
        cleaned = re.sub(r",\s*]", "]", cleaned)
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return {"_parse_error": True, "_raw_output": raw_output}


def merge_mrz_and_vlm(mrz_result: Dict[str, Any], vlm_fields: Dict[str, Any]) -> Dict[str, Any]:
    '''La MRZ prime sur les champs qu'elle couvre (source deterministe + checksum valide).'''
    merged = dict(vlm_fields)
    if mrz_result.get("mrz_present") and mrz_result.get("fields"):
        f = mrz_result["fields"]
        if f.get("surname"):
            merged["nom"] = f["surname"]
        if f.get("given_names"):
            merged["prenom"] = f["given_names"]
        if f.get("document_number"):
            merged["numero_piece"] = f["document_number"]
        if f.get("birth_date"):
            merged["date_naissance"] = f["birth_date"]
        if f.get("expiry_date"):
            merged["date_expiration"] = f["expiry_date"]
    merged["mrz_present"] = mrz_result.get("mrz_present", False)
    merged["mrz_type"] = mrz_result.get("mrz_type")
    return merged


## 10. Pipeline principal — extraction par client

Pour chaque dossier client :
1. Repérage des 2 PDF cibles.
2. Pour `identite` : preprocessing + détection MRZ + détection pays + prompt adapté + VLM + fusion.
3. Pour `domicile` : preprocessing + détection pays (via le doc identité si déjà connu, sinon via OCR
   du domicile lui-même) + prompt adapté + VLM (pas de MRZ sur un justificatif de domicile).

In [ ]:

def process_identity_doc(pdf_path: Path, vlm_engine: "VLMEngine") -> Dict[str, Any]:
    pages = pdf_to_images(pdf_path, dpi=CONFIG["pdf_dpi"])
    # rotation corrigée avant toute analyse (en-tête + MRZ + VLM), comme demandé
    img = preprocess_document_image(pages[0])  # recto (page 1) — adapter si recto/verso sur 2 pages

    # 1) Type de document via l'en-tête ("s'agit-il bien d'une pièce d'identité ?")
    is_id_header = detect_document_kind_from_header(img)

    # 2) Carte biométrique ? (présence d'une zone MRZ) + parsing (règles DZ dédiées ou génériques)
    mrz_result = detect_and_parse_mrz(img)

    # 3) Pays d'émission (s'appuie en priorité sur le code pays lu dans la MRZ)
    is_algeria = detect_country_is_algeria(img, mrz_result)
    prompt = build_prompt("identite", is_algeria)

    raw_output = vlm_engine.generate(img, prompt)
    vlm_fields = safe_parse_json(raw_output)
    merged = merge_mrz_and_vlm(mrz_result, vlm_fields)

    merged["document_kind_from_header"] = "identite" if is_id_header else "inconnu"
    merged["is_biometric"] = mrz_result.get("is_biometric", False)
    merged["checksum_valid"] = mrz_result.get("checksum_valid")
    merged["pays_detecte"] = "DZ" if is_algeria else "AUTRE"
    merged["document_type"] = "identite"
    return merged


def process_residence_doc(pdf_path: Path, vlm_engine: "VLMEngine", is_algeria_hint: Optional[bool]) -> Dict[str, Any]:
    pages = pdf_to_images(pdf_path, dpi=CONFIG["pdf_dpi"])
    img = preprocess_document_image(pages[0])

    if is_algeria_hint is None:
        is_algeria = detect_country_is_algeria(img, {"fields": None})
    else:
        is_algeria = is_algeria_hint

    prompt = build_prompt("domicile", is_algeria)
    raw_output = vlm_engine.generate(img, prompt)
    vlm_fields = safe_parse_json(raw_output)
    vlm_fields["mrz_present"] = False
    vlm_fields["mrz_type"] = None
    vlm_fields["pays_detecte"] = "DZ" if is_algeria else "AUTRE"
    vlm_fields["document_type"] = "domicile"
    return vlm_fields


def run_pipeline(extracted_root: str, vlm_engine: "VLMEngine") -> pd.DataFrame:
    rows = []
    client_folders = list_client_folders(extracted_root)

    for client_dir in client_folders:
        client_id = client_dir.name
        found = find_target_pdfs(client_dir, CONFIG["target_docs"], CONFIG["filename_match_threshold"])

        record_id, record_dom = {}, {}
        is_algeria_hint = None

        if found.get("identite"):
            try:
                record_id = process_identity_doc(found["identite"], vlm_engine)
                is_algeria_hint = (record_id.get("pays_detecte") == "DZ")
            except Exception as e:
                record_id = {"_error": str(e), "document_type": "identite"}
        else:
            record_id = {"_error": "PDF identite introuvable", "document_type": "identite"}

        if found.get("domicile"):
            try:
                record_dom = process_residence_doc(found["domicile"], vlm_engine, is_algeria_hint)
            except Exception as e:
                record_dom = {"_error": str(e), "document_type": "domicile"}
        else:
            record_dom = {"_error": "PDF domicile introuvable", "document_type": "domicile"}

        for rec in (record_id, record_dom):
            rec["client_id"] = client_id
            rows.append(rec)

    return pd.DataFrame(rows)


## 11. Exécution du pipeline

In [ ]:

# 1) Dézippage
extracted_root = extract_zip(CONFIG["zip_path"], CONFIG["work_dir"])

# 2) Chargement du modèle choisi
vlm_engine = VLMEngine(CONFIG["model_choice"], CONFIG["model_paths"][CONFIG["model_choice"]])

# 3) Exécution
df_extraction = run_pipeline(extracted_root, vlm_engine)

# 4) Sauvegarde brute
extraction_out_path = os.path.join(CONFIG["output_dir"], "extraction_kyc.csv")
df_extraction.to_csv(extraction_out_path, index=False, encoding="utf-8-sig")
df_extraction.head(20)


## 12. Réconciliation avec `tiers.csv`

Normalisation des chaînes (accents, casse, espaces) puis comparaison floue champ par champ.
Toute similarité en dessous du seuil `fuzzy_match_threshold` est signalée comme incohérence.
Le mapping des colonnes de `tiers.csv` (`CONFIG["tiers_columns_mapping"]`) est à adapter au
fichier réel si les noms de colonnes diffèrent.

In [ ]:

def normalize_str(x) -> str:
    if pd.isna(x) or x is None:
        return ""
    return unidecode(str(x)).upper().strip()


def compare_field(extracted_val, tiers_val, threshold: int) -> Dict[str, Any]:
    e, t = normalize_str(extracted_val), normalize_str(tiers_val)
    if not e and not t:
        return {"score": None, "status": "N/A"}
    if not e or not t:
        return {"score": 0, "status": "MANQUANT"}
    score = fuzz.token_sort_ratio(e, t)
    status = "OK" if score >= threshold else "INCOHERENCE"
    return {"score": score, "status": status}


def build_reconciliation_report(df_extraction: pd.DataFrame, tiers_csv_path: str,
                                 mapping: Dict[str, str], threshold: int) -> pd.DataFrame:
    tiers = pd.read_csv(tiers_csv_path, dtype=str)

    # On ne réconcilie que le document 'identite' (source la plus fiable / la plus structurée),
    # le document 'domicile' peut être ajouté de la même façon pour le champ 'adresse'.
    df_id = df_extraction[df_extraction["document_type"] == "identite"].copy()
    df_dom = df_extraction[df_extraction["document_type"] == "domicile"].copy()

    merged = df_id.merge(
        tiers, left_on="client_id", right_on=mapping["id_tiers"], how="left", suffixes=("_extrait", "_tiers")
    )
    # on rattache aussi l'adresse extraite du justificatif de domicile
    merged = merged.merge(
        df_dom[["client_id", "adresse"]].rename(columns={"adresse": "adresse_domicile_extrait"}),
        on="client_id", how="left"
    )

    fields_to_compare = [
        ("nom", mapping["nom"]),
        ("prenom", mapping["prenom"]),
        ("date_naissance", mapping["date_naissance"]),
        ("numero_piece", mapping["numero_piece"]),
    ]

    report_rows = []
    for _, row in merged.iterrows():
        report = {"client_id": row["client_id"], "mrz_present": row.get("mrz_present", False)}
        incoherences = []
        for extracted_col, tiers_col in fields_to_compare:
            cmp = compare_field(row.get(extracted_col), row.get(tiers_col), threshold)
            report[f"{extracted_col}_score"] = cmp["score"]
            report[f"{extracted_col}_status"] = cmp["status"]
            if cmp["status"] == "INCOHERENCE":
                incoherences.append(extracted_col)

        # adresse : on compare le meilleur des deux champs adresse extraits (identité / domicile)
        adr_cmp_dom = compare_field(row.get("adresse_domicile_extrait"), row.get(mapping["adresse"]), threshold)
        report["adresse_score"] = adr_cmp_dom["score"]
        report["adresse_status"] = adr_cmp_dom["status"]
        if adr_cmp_dom["status"] == "INCOHERENCE":
            incoherences.append("adresse")

        report["nb_incoherences"] = len(incoherences)
        report["champs_incoherents"] = ", ".join(incoherences) if incoherences else ""
        report["decision"] = "A REVOIR" if incoherences else "OK"
        report_rows.append(report)

    return pd.DataFrame(report_rows)


df_reconciliation = build_reconciliation_report(
    df_extraction, CONFIG["tiers_csv_path"], CONFIG["tiers_columns_mapping"], CONFIG["fuzzy_match_threshold"]
)

reconciliation_out_path = os.path.join(CONFIG["output_dir"], "rapport_reconciliation.csv")
df_reconciliation.to_csv(reconciliation_out_path, index=False, encoding="utf-8-sig")

print(f"Clients traités        : {df_reconciliation['client_id'].nunique()}")
print(f"Dossiers 'A REVOIR'     : {(df_reconciliation['decision'] == 'A REVOIR').sum()}")
print(f"Dossiers OK             : {(df_reconciliation['decision'] == 'OK').sum()}")
df_reconciliation.sort_values('nb_incoherences', ascending=False).head(20)


## 13. Notes, limites et points à ajuster avant mise en production

- **Colonnes de `tiers.csv`** : adapter `CONFIG["tiers_columns_mapping"]` aux noms réels de colonnes.
- **PaddleOCR-VL / InternVL3** : les adaptateurs `VLMEngine` sont écrits selon les patterns d'API
  habituels (`trust_remote_code=True`, méthode `.chat()` pour InternVL). Vérifier le `README.md` /
  `modeling_*.py` exact déposé sur le model hub interne (`/domino/edv/modelhub/.../main`) car ces
  détails peuvent varier d'une révision à l'autre — en particulier pour PaddleOCR-VL, plus récent.
- **Recto/verso** : le pipeline traite ici la première page du PDF comme recto ; si les CNI incluent
  systématiquement un verso utile (adresse, empreintes...), ajouter le traitement de `pages[1]`
  et fusionner les champs des deux faces.
- **Seuils** (`filename_match_threshold`, `fuzzy_match_threshold`) sont à calibrer sur un échantillon
  réel étiqueté pour éviter faux positifs/négatifs.
- **Performance** : pour un gros volume de dossiers, envisager un traitement par batch et/ou
  quantification du modèle (bitsandbytes / AWQ) selon la charge GPU disponible.
- **Qualité MRZ** : sur des scans très dégradés, l'OCR de la zone MRZ peut échouer même si la MRZ
  existe physiquement — dans ce cas `mrz_present=False` et le VLM prend le relais automatiquement,
  ce qui est le comportement voulu (dégradation progressive plutôt que blocage).
- **RGPD / sécurité** : les dossiers dézippés en local (`CONFIG["work_dir"]`) contiennent des
  données personnelles sensibles — prévoir un nettoyage (`shutil.rmtree`) après traitement et un
  chiffrement au repos si l'environnement l'exige.
